In [28]:
import requests
import tkinter as tk
from tkinter import messagebox
from PIL import Image, ImageTk
from io import BytesIO

# SEARCH FUNCTION (iTunes API)
# ==========================
def search_song():
    query = search_entry.get().strip()

    if not query:
        messagebox.showwarning("Input Error", "Please enter a search keyword.")
        return

    url = "https://itunes.apple.com/search"
    params = {
        "term": query,
        "entity": "song",
        "limit": 1
    }

    try:
        response = requests.get(url, params=params)
        data = response.json()
    except Exception as e:
        messagebox.showerror("Error", f"API Error:\n{e}")
        return

    results = data.get("results", [])

    if not results:
        messagebox.showinfo("No Results", "No songs found.")
        return

    track = results[0]

    # Fetch data
    title = track.get("trackName", "Unknown Title")
    artist = track.get("artistName", "Unknown Artist")
    album = track.get("collectionName", "Unknown Album")
    release_date = track.get("releaseDate", "Unknown Date")[:10]
    image_url = track.get("artworkUrl100")

    # Update UI text
    title_var.set(f"🎵 Title: {title}")
    artist_var.set(f"👤 Artist: {artist}")
    album_var.set(f"💿 Album: {album}")
    date_var.set(f"📅 Release Date: {release_date}")

    # Load cover art
    if image_url:
        try:
            # Get high-resolution version (600px)
            image_url = image_url.replace("100x100bb", "600x600bb")

            img_data = requests.get(image_url).content
            img = Image.open(BytesIO(img_data))
            img = img.resize((250, 250))
            img_tk = ImageTk.PhotoImage(img)

            cover_label.config(image=img_tk)
            cover_label.image = img_tk
        except:
            cover_label.config(text="(Image Load Failed)")
    else:
        cover_label.config(text="No Cover Art Found")


